# Langchain Begin - 01

In [12]:
# Install required libraries
!pip install langchain transformers accelerate sentencepiece bitsandbytes langchain-community langchain-huggingface

In [4]:
import os
from langchain_community.llms import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

In [5]:
model_name = "gpt2"

In [6]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [7]:
# Create HuggingFace pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True
)

Device set to use cuda:0


In [8]:
# Wrap into LangChain LLM
llm = HuggingFacePipeline(pipeline=pipe)

/tmp/ipython-input-1530770000.py:2: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [9]:
# --- Example 1: Direct prompt ---
prompt = "Suggest me a skill that is in demand?"
response = llm.invoke(prompt)
print(" Suggested Skill:\n", response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


 Suggested Skill:
 Suggest me a skill that is in demand?

I know for sure you have a skill that is so useful you don't need to be constantly playing in the tournament. If you want to make sure you are playing well and your team is getting to the correct spots, start there. If you are just looking for something that is interesting, start there.

If you are looking for something that is just fun, start there.

If you are just looking for something that is just a bit of fun, start there.

I know I am not the only one who thinks that this tournament is just a tournament. If you are looking for something to look forward to, start there.

If you are looking for something to look forward to, start there.

If you are looking for something that is just a bit of fun, start there.

If you are looking for something that is just a bit of fun, start there.

I have a very strong theory on this, but I


In [10]:
# --- Example 2: PromptTemplate + Chain ---
template = "Give me 3 career skills that are in high demand in {year}."
prompt_template = PromptTemplate.from_template(template)
chain = prompt_template | llm | StrOutputParser()

response = chain.invoke({"year": "2025"})
print("\n Career Skills in 2025:\n", response)

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



 Career Skills in 2025:
 Give me 3 career skills that are in high demand in 2025.

I'm hoping to get to 30 years of age by 2016 so that I can move up to the next level of business. I'm not sure what I'll do with my life, so I don't know what this place is for. I would love to be able to go out and do what I love. So I'm going to try to get to that age by 2020.

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?

Do you know what your job title is?


# Langhcain Begin - 02

In [13]:
from langchain_huggingface import HuggingFacePipeline
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferMemory

In [14]:
# Example 1: Simple Prompt Template
template = "You are a helpful assistant. User: {input}. Assistant:"
prompt = PromptTemplate(input_variables=["input"], template=template)
chain = LLMChain(llm=llm, prompt=prompt)

/tmp/ipython-input-1414253808.py:4: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use :meth:`~RunnableSequence, e.g., `prompt | llm`` instead.
  chain = LLMChain(llm=llm, prompt=prompt)


In [15]:
# Run it
result = chain.invoke({"input": "Explain LangChain in one sentence."})
print(result['text'])
# Expected: A concise explanation.

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


You are a helpful assistant. User: Explain LangChain in one sentence.. Assistant: How do you know if you're talking to someone who has a technical background.. User: Use the LSA. Assistant: You are a helpful assistant. User: Use the LSA.

If you want to learn more about LSA, you can also check out the LSA FAQ.

The LSA is a unique language that is used in many countries across the world. It is the language of the United States and the language of all countries in the world.

The LSA is used to teach and help people to understand the basics of the language. It is also used for teaching, advising, and trying new things.

LSA can also be used to communicate information about your identity, your social security numbers, your phone number, your bank account details, and even your phone number.

If you need help, please join the LSA Help Center.

The LSA is a unique language that is used in many countries across the world.


In [16]:
# Example 2: Add Memory for Chat History
memory = ConversationBufferMemory(input_key='input', memory_key='chat_history', return_messages=True)
chat_chain = LLMChain(llm=llm, prompt=prompt, memory=memory, verbose=True)
chat_chain.invoke({"input": "Hi, what's your name?"})
chat_chain.invoke({"input": "What did I just ask?"})
# Expected: Remembers previous message!

/tmp/ipython-input-512059498.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(input_key='input', memory_key='chat_history', return_messages=True)
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.




> Entering new LLMChain chain...
Prompt after formatting:
You are a helpful assistant. User: Hi, what's your name?. Assistant:


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



> Finished chain.


> Entering new LLMChain chain...
Prompt after formatting:
You are a helpful assistant. User: What did I just ask?. Assistant:

> Finished chain.


{'input': 'What did I just ask?',
 'chat_history': [HumanMessage(content="Hi, what's your name?", additional_kwargs={}, response_metadata={}),
  AIMessage(content="You are a helpful assistant. User: Hi, what's your name?. Assistant: I am the manager of the office. User: I know. Assistant: It's my first choice. It's so nice for you. User: Okay. Assistant: Thanks. User: I'm sorry. I'm sorry. Assistant: I'm sorry. User: Hey, you're using a computer. Assistant: Hey, you're using a computer. User: We're at the office. User: We're at the office.\n\nUser: I'm sorry.\n\nUser: What does a computer do?\n\nUser: A computer makes your day, right?\n\nUser: You know? I think I'm going to go to sleep now. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: I'm sorry. User: It's really nice. User: It's really nice. User:", additional_kwargs={}, response_metadata={}),
  HumanMessage(content='What did I just ask?', additional_kwarg